### OpenWeather API

- Create an account [here](https://openweathermap.org/). You will use their API to get weather forecast data
- After you have created the free account, request an API key. The key enables you to connect to their API and pull weather data.
- Helpful video to understand API calls with Python [here](https://www.youtube.com/watch?v=_jBeFdqnNAU)

In [0]:
# Execute this cell
import pandas as pd
import numpy as np
import requests

# this cell includes the dataframes you'll use later
# creating the pandas dataframes
# creating an empty table to store errors
weather_error_logs = pd.DataFrame(columns=['ErrorCode','ErrorMessage','GivenCity','GivenLatitude','GivenLongtitude'])


### Weather Function
- Create a function which will make requests to the OpenWeather [Current Weather Data API](https://openweathermap.org/current) to get weather information using the latitude and longitude coordinates for a city. 
- Parse the response from the API request and extract the following information about a city: <code>id,name,country,main description,temperature,minimum temperature,maximum temperature,feels_like,humidity and wind speed<code>.
- The function should return a dictionary of the format:
  ```
    {
      "id": 802,
      "city": "Skopje",
      "latitude": 42,
      "longtitude": 21.4333,
      "country": 25,
      "description": "scattered clouds",
      "temp":290.97,
      "temp_min": 293.97,
      "temp_max": 293.97,
      "feels_like": 293.23,
      "humidity": 43,
      "wind_speed": 2.06   
     }
  ```
 - Design the function in a way that it should work for any given pair of values for latittude and longitute coordinates.

In [0]:
import pandas as pd
import requests

# API key
weather_api_key = "27bda93880ee172a5b66ef4156653b1c"

# error logs dataframe (MUST MATCH GIVEN STRUCTURE)
weather_error_logs = pd.DataFrame(columns=[
    'ErrorCode',
    'ErrorMessage',
    'GivenCity',
    'GivenLatitude',
    'GivenLongtitude'
])
#EXTRACT
def get_weather_data(city, lat, lon, weather_api_key):
    # url to give me weather data for a given city, lat, lon
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={weather_api_key}"
    
    try:
        # You send request to API server. Server responds with data
        response = requests.get(url)
        # API response is converted into a Python dictionary
        data = response.json()

        # ERROR CASE
        if response.status_code != 200:
            weather_error_logs.loc[len(weather_error_logs)] = [
                response.status_code,
                data.get("message", "error"),
                city,
                lat,
                lon
            ]

            return {
                "id": None,
                "latitude": lat,
                "longtitude": lon,
                "city": city,
                "country": None,
                "description": data.get("message", "error"),
                "temp": None,
                "temp_min": None,
                "temp_max": None,
                "feels_like": None,
                "humidity": None,
                "wind_speed": None
            }

        # SUCCESS CASE
        return {
            "id": data["weather"][0]["id"],
            "latitude": lat,
            "longtitude": lon,
            "city": data["name"],
            "country": data["sys"]["country"],
            "description": data["weather"][0]["description"],
            "temp": data["main"]["temp"],
            "temp_min": data["main"]["temp_min"],
            "temp_max": data["main"]["temp_max"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "wind_speed": data["wind"]["speed"]
        }

    except Exception as e:
        # EXCEPTION CASE
        weather_error_logs.loc[len(weather_error_logs)] = [
            "EXCEPTION",
            str(e),
            city,
            lat,
            lon
        ]

        return {
            "id": None,
            "latitude": lat,
            "longtitude": lon,
            "city": city,
            "country": None,
            "description": str(e),
            "temp": None,
            "temp_min": None,
            "temp_max": None,
            "feels_like": None,
            "humidity": None,
            "wind_speed": None
        }

In [0]:
# Run this cell to check your function
city,lat,lon = 'Skopje', 42, 21.4333

answer = get_weather_data(city,lat,lon,weather_api_key)

# checking if you are returning a dictionary
assert isinstance(answer,dict), "You are not returning a dictionary"

# checking if the returned dictionary contains all the right keys
weather_keys = [ "id","latitude","longtitude", "city","country","description","temp","temp_min","temp_max","feels_like","humidity","wind_speed"]
assert all(key in answer for key in weather_keys),"You are missing keys"
print(answer)

#### Handle failures in weather function 

- A failure in this case as an example would be considered passing wrong values for latitude and longtitude. For this reason, modify the weather function you developed before to gracefully fail if wrong values for latitude and longtitude are passed. 
- The function should capture the response error message from the API call. In these cases, record this information as a new entry into the dataframe **weather_error_logs**. Explore the structure of this dataframe to see what information you need to capture and what columns you need to populate. If required, process the response further to populate the columns accordingly.

In [0]:
import requests

def get_weather_data(city, lat, lon, weather_api_key):
    global weather_error_logs  # ✅ important fix

    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={weather_api_key}"
    
    try:
        response = requests.get(url)
        data = response.json()

        # HANDLE API ERROR (wrong lat/lon, bad request, etc.)
        if response.status_code != 200:
            weather_error_logs.loc[len(weather_error_logs)] = [
                response.status_code,
                data.get("message", "Unknown error"),
                city,
                lat,
                lon
            ]

            return {
                "id": None,
                "latitude": lat,
                "longtitude": lon,  # keep as assignment requires
                "city": city,
                "country": None,
                "description": data.get("message", "error"),
                "temp": None,
                "temp_min": None,
                "temp_max": None,
                "feels_like": None,
                "humidity": None,
                "wind_speed": None
            }

        # SUCCESS CASE
        return {
            "id": data["weather"][0]["id"],
            "latitude": lat,
            "longtitude": lon,
            "city": data["name"],
            "country": data["sys"]["country"],
            "description": data["weather"][0]["description"],
            "temp": data["main"]["temp"],
            "temp_min": data["main"]["temp_min"],
            "temp_max": data["main"]["temp_max"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "wind_speed": data["wind"]["speed"]
        }

    except Exception as e:
        # HANDLE RUNTIME ERRORS
        weather_error_logs.loc[len(weather_error_logs)] = [
            "EXCEPTION",
            str(e),
            city,
            lat,
            lon
        ]

        return {
            "id": None,
            "latitude": lat,
            "longtitude": lon,
            "city": city,
            "country": None,
            "description": str(e),
            "temp": None,
            "temp_min": None,
            "temp_max": None,
            "feels_like": None,
            "humidity": None,
            "wind_speed": None
        }

In [0]:
result = get_weather_data("Skopje", 42, 21.4333, weather_api_key)
print(result)

#### Get weather forecast for new locations
Below you are given a pandas dataframe **weather_df** holding data for new cities and their latitude and longtitude values. 
- For each city in the dataframe, by using the function you developed before, get the weather information and add it to the dataframe. Explore the structure of the dataframe and populate the columns accordingly.

In [0]:
# creating the pandas dataframes
import numpy as np
weather_df = pd.DataFrame(
    data=np.array([
        ["Skopje",42,21.4333],
        ["Tetovo",7958,21.4333], #wrong lat&long
        ["London",51.5085,-0.1257],
        ["Tokyo",35.6895,139.6917],
        ["New York",40.7143,-74.006],
        ["Berlin",52.5244,13.4105]
    ]),
    columns=['city','latitude','longtitude']
)

# use NaN instead of empty string
for col in ["id","country","description","temp","temp_min","temp_max","feels_like","humidity","wind_speed"]:
    weather_df[col] = np.nan

print('Weather Dataframe Before:\n', weather_df.head())

# LOOP
for index, row in weather_df.iterrows():
    city = row["city"]
    lat = float(row["latitude"])
    lon = float(row["longtitude"])

    result = get_weather_data(city, lat, lon, weather_api_key)

    # Skip rows with error (already logged)
    if result["id"] is None:
        continue

    weather_df.loc[index, "id"] = result["id"]
    weather_df.loc[index, "country"] = result["country"]
    weather_df.loc[index, "description"] = result["description"]
    weather_df.loc[index, "temp"] = result["temp"]
    weather_df.loc[index, "temp_min"] = result["temp_min"]
    weather_df.loc[index, "temp_max"] = result["temp_max"]
    weather_df.loc[index, "feels_like"] = result["feels_like"]
    weather_df.loc[index, "humidity"] = result["humidity"]
    weather_df.loc[index, "wind_speed"] = result["wind_speed"]

print('Weather Dataframe After:\n', weather_df.head())

**Save the dataframes as tables**

- First, convert the two pandas dataframes `weather_df, weather_error_logs` into Spark dataframes. This [guide](https://docs.databricks.com/pandas/pyspark-pandas-conversion.html) shows how you can achieve that.
- Save the newly created `weather_df, weather_error_logs` Spark DataFrames as managed tables using `.write.mode("overwrite").saveAsTable()`.

> **💡 Tip:** When converting a Pandas DataFrame to a Spark DataFrame, make sure each column has a consistent data type (e.g., all numeric or all string). Mixed types in a column will cause an Arrow conversion error. Use `pd.to_numeric()` to cast columns that should be numeric.

In [0]:

# # Write your code here

# # Step 1: Convert Pandas DataFrame to Spark DataFrame

# # Step 2: Write Spark DataFrame to Parquet file in DBFS

# # Step 3: Register the table using CREATE TABLE statement



# Step 1: Convert Pandas DataFrame to Spark DataFrame
numeric_cols = ["id", "temp", "temp_min", "temp_max", "feels_like", "humidity", "wind_speed"]

# Transform to numeric values; errors become NaN instead of crashing
for col in numeric_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')

# Convert Pandas DataFrames to Spark DataFrames
spark_weather_df = spark.createDataFrame(weather_df)
spark_error_df = spark.createDataFrame(weather_error_logs)


# Step 2: Write Spark DataFrame to Parquet file in DBFS
spark_weather_df.write.mode("overwrite").parquet("dbfs:/FileStore/tables/weather_df")
spark_error_df.write.mode("overwrite").parquet("dbfs:/FileStore/tables/weather_error_logs")


# Step 3: Register the tables using CREATE TABLE statement
spark.sql("""
    CREATE TABLE IF NOT EXISTS weather_df
    USING PARQUET
    LOCATION 'dbfs:/FileStore/tables/weather_df'
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS weather_error_logs
    USING PARQUET
    LOCATION 'dbfs:/FileStore/tables/weather_error_logs'
""")


# # Step 4: Verify
# spark.sql("SELECT * FROM weather_df").show()
# spark.sql("SELECT * FROM weather_error_logs").show()

# # Remove old tables first
# spark.sql("DROP TABLE IF EXISTS weather_df")
# spark.sql("DROP TABLE IF EXISTS weather_error_logs")

# numeric_cols = ["id", "temp", "temp_min", "temp_max", "feels_like", "humidity", "wind_speed"]

# for col in numeric_cols:
#     weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')

# # Fix mixed types
# weather_error_logs["ErrorCode"] = weather_error_logs["ErrorCode"].astype(str)
# weather_error_logs["ErrorMessage"] = weather_error_logs["ErrorMessage"].astype(str)
# weather_error_logs["GivenCity"] = weather_error_logs["GivenCity"].astype(str)

# # Convert to Spark
# spark_weather_df = spark.createDataFrame(weather_df)
# spark_error_df = spark.createDataFrame(weather_error_logs)

# # Save tables
# spark_weather_df.write.mode("overwrite").saveAsTable("weather_df")
# spark_error_df.write.mode("overwrite").saveAsTable("weather_error_logs")

# # Verify
# spark.sql("SELECT * FROM weather_df").show()
# spark.sql("SELECT * FROM weather_error_logs").show()

In [0]:
%sql
-- Displaying the tables
-- select * from `weather_data`
select * from `weather_error_logs`